# Contextual LogEI Campaign

This notebook demonstrates BO Forge's conservative contextual BO workflow. The context variable remains a normal CSV variable, but suggestion generation fixes it through `context_values` while optimizing the decision variables.

In [ ]:
from pathlib import Path
import math
import shutil

import pandas as pd

from bo_forge import CampaignSession

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
CONFIG_PATH = PROJECT_ROOT / "configs" / "16_contextual_logei.yaml"
SEED_LOG_PATH = PROJECT_ROOT / "examples" / "16_contextual_logei_campaign_log.csv"
WORKING_LOG_PATH = PROJECT_ROOT / "examples" / "16_contextual_logei_working_log.csv"
LATEST_SUGGESTIONS_PATH = PROJECT_ROOT / "examples" / "16_contextual_logei_latest_suggestions.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
TARGET_OBSERVED_ROWS = 15

REPORTS_DIR.mkdir(exist_ok=True)
shutil.copyfile(SEED_LOG_PATH, WORKING_LOG_PATH)

Load and validate the campaign. The seed log is left unchanged; all mutation happens against the ignored working log.

In [ ]:
campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)
campaign.validate()
campaign.summary()

In [ ]:
campaign.context_summary()

In [ ]:
campaign.next_action()

The synthetic response below is deterministic and local. It is only a teaching stand-in for a real experiment measured at the requested context.

In [ ]:
def simulate_yield(row: pd.Series) -> float:
    loading = float(row["catalyst_loading"])
    temperature = float(row["reaction_temperature"])
    acidity = float(row["feedstock_acidity"])
    solvent_bonus = {"MeCN": 0.08, "EtOH": 0.03, "Water": -0.04}[str(row["solvent"])]
    loading_term = 0.95 + 0.75 * loading - 1.10 * (loading - 0.55) ** 2
    temperature_term = -0.00009 * (temperature - 92.0) ** 2
    context_term = 0.22 * acidity - 0.35 * (acidity - 0.55) ** 2
    smooth_variation = 0.025 * math.sin(7.0 * loading + 0.04 * temperature + 2.0 * acidity)
    return round(loading_term + temperature_term + context_term + solvent_bonus + smooth_variation, 6)

Run a sequential loop over explicit context values. `suggest_next()` remains non-mutating; rows are written only after the explicit `append_suggestions()` call.

In [ ]:
context_cycle = [0.25, 0.5, 0.75]
records = []

while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    acidity = context_cycle[len(records) % len(context_cycle)]
    context_values = {"feedstock_acidity": acidity}
    suggestions = campaign.suggest_next(context_values=context_values)
    suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
    campaign.append_suggestions(suggestions)
    for row_id in suggestions["row_id"]:
        row = campaign.df.loc[campaign.df["row_id"] == row_id].iloc[0]
        value = simulate_yield(row)
        campaign.mark_observed(row_id=row_id, objective_value=value)
        records.append({"row_id": row_id, "feedstock_acidity": acidity, "yield_score": value})

pd.DataFrame(records).tail()

Reload and inspect the contextual campaign state.

In [ ]:
campaign.reload()
campaign.summary()

In [ ]:
campaign.context_summary()

Export the text report and the standard progress, diagnostics, and context diagnostics plots.

In [ ]:
campaign.export_report(REPORTS_DIR / "16_contextual_logei_report.txt")
campaign.plot_progress(save_path=REPORTS_DIR / "16_contextual_logei_progress.png")
campaign.plot_diagnostics(save_path=REPORTS_DIR / "16_contextual_logei_diagnostics.png")
campaign.plot_context_diagnostics(save_path=REPORTS_DIR / "16_contextual_logei_context_diagnostics.png")

CLI equivalent for a dry-run contextual suggestion:

```bash
python -m bo_forge suggest \
  --config configs/16_contextual_logei.yaml \
  --log examples/16_contextual_logei_campaign_log.csv \
  --context feedstock_acidity=0.25 \
  --batch-size 1
python -m bo_forge context-summary \
  --config configs/16_contextual_logei.yaml \
  --log examples/16_contextual_logei_campaign_log.csv
```

Limitations: contextual BO remains single-objective LogEI/qLogEI only in this notebook. Contextual cost/review support is covered by notebook 20; contextual multi-objective, structured, multi-fidelity, qLogNEI/qLogNEHVI, and replicate-aware workflows remain deferred.